In [1]:
!nvidia-smi


Sun Mar 29 15:46:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/saccharomycetes/mllms_know.git



fatal: destination path 'mllms_know' already exists and is not an empty directory.


In [3]:
%cd /content/mllms_know
!pip install -r requirements.txt


/content/mllms_know


In [4]:
%cd /content/mllms_know/transformers
!pip install -e .
%cd /content/mllms_know


/content/mllms_know/transformers
Obtaining file:///content/mllms_know/transformers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.8 MB/s eta 0:00:00
  Building editable for transformers (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.47.0.dev0-0.editable-py3-none-any.whl size=17428 sha256=738d410f46561dbba20f147be938dbc2c3b0d7b3c3a440ef8131135e4ce938f7
  Stored in directory: /tmp/pip-ephem-wheel-cache-fbbt49cl/wheels/1c/22/58/56e810460716a81d9d431990217451c4330ac0196c9d94c6b4
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successful

In [5]:
from pathlib import Path
import re

# run.py
run_path = Path("/content/mllms_know/run.py")
run_text = run_path.read_text()

run_text = re.sub(
    r"from transformers import AutoProcessor, LlavaForConditionalGeneration, InstructBlipProcessor, InstructBlipForConditionalGeneration, Qwen2_5_VLForConditionalGeneration",
    """from transformers import AutoProcessor, LlavaForConditionalGeneration, InstructBlipProcessor, InstructBlipForConditionalGeneration

try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError:
    Qwen2_5_VLForConditionalGeneration = None""",
    run_text,
)

run_path.write_text(run_text)

# utils.py
utils_path = Path("/content/mllms_know/utils.py")
utils_text = utils_path.read_text()
utils_text = re.sub(
    r"from qwen_vl_utils import process_vision_info",
    """try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None""",
    utils_text,
)
utils_path.write_text(utils_text)

print("Patched run.py and utils.py")


Patched run.py and utils.py


In [6]:
!mkdir -p /content/mllms_know/data/textvqa/images
!wget https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip -P /content/mllms_know/data/textvqa/images
!unzip -q /content/mllms_know/data/textvqa/images/train_val_images.zip -d /content/mllms_know/data/textvqa/images
!rm /content/mllms_know/data/textvqa/images/train_val_images.zip
!mv /content/mllms_know/data/textvqa/images/train_images/* /content/mllms_know/data/textvqa/images/
!rm -r /content/mllms_know/data/textvqa/images/train_images
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json -P /content/mllms_know/data/textvqa


--2026-03-29 15:48:07--  https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.62, 13.249.182.39, 13.249.182.33, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7072297970 (6.6G) [application/zip]
Saving to: ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’

train_val_images.zi 100%[===================>]   6.59G  79.2MB/s    in 83s     

2026-03-29 15:49:30 (81.6 MB/s) - ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’ saved [7072297970/7072297970]

--2026-03-29 15:50:53--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.39, 13.249.182.62, 13.249.182.33, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.39|:443... connected.
HTTP request sent, awaiting res

In [7]:
!pip install bitsandbytes


In [8]:
from pathlib import Path

path = Path("/content/mllms_know/run.py")
text = path.read_text()

old = """    elif args.model == 'blip':
        model = InstructBlipForConditionalGeneration.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).to(args.device)
        processor = InstructBlipProcessor.from_pretrained(args.model_id)"""

new = """    elif args.model == 'blip':
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = InstructBlipForConditionalGeneration.from_pretrained(
            args.model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            quantization_config=quantization_config,
            device_map="auto",
        )
        processor = InstructBlipProcessor.from_pretrained(args.model_id)"""

text = text.replace(old, new)
path.write_text(text)

print("Patched BLIP loading to 4-bit")


Patched BLIP loading to 4-bit


In [9]:
from pathlib import Path

for file in [
    "/content/mllms_know/run.py",
    "/content/mllms_know/blip_methods.py",
]:
    p = Path(file)
    t = p.read_text().replace("torch.bfloat16", "torch.float16")
    p.write_text(t)

print("Patched bf16 -> fp16 for BLIP path")


Patched bf16 -> fp16 for BLIP path


In [14]:
from pathlib import Path

path = Path("/content/mllms_know/blip_methods.py")
text = path.read_text()

text = text.replace(
    "lm_att = lm_atts[LM_LAYER][0, :, -1, :NUM_IMG_TOKENS]",
    "lm_att = lm_atts[LM_LAYER][0, :, -1, :QFORMER_IMG_TOKENS]"
)

text = text.replace(
    "lm_grad = lm_att_grads[LM_LAYER][0, :, -1, :NUM_IMG_TOKENS]",
    "lm_grad = lm_att_grads[LM_LAYER][0, :, -1, :QFORMER_IMG_TOKENS]"
)

text = text.replace(
    "lm_atts = lm_atts[LM_LAYER][0, :, -1, :NUM_IMG_TOKENS].mean(dim=0).unsqueeze(0).unsqueeze(0)",
    "lm_atts = lm_atts[LM_LAYER][0, :, -1, :QFORMER_IMG_TOKENS].mean(dim=0).unsqueeze(0).unsqueeze(0)"
)

text = text.replace(
    "general_lm_atts = general_lm_atts[LM_LAYER][0, :, -1, :NUM_IMG_TOKENS].mean(dim=0).unsqueeze(0).unsqueeze(0)",
    "general_lm_atts = general_lm_atts[LM_LAYER][0, :, -1, :QFORMER_IMG_TOKENS].mean(dim=0).unsqueeze(0).unsqueeze(0)"
)

path.write_text(text)
print("Patched blip_methods.py")


Patched blip_methods.py


In [15]:
import json

src = "/content/mllms_know/data/textvqa/TextVQA_0.5.1_val.json"
dst = "/content/mllms_know/data/textvqa/data.json"
backup = "/content/mllms_know/data/textvqa/data_full_backup.json"

with open(src) as f:
    datas = json.load(f)

new_datas = []
for data_id, data in enumerate(datas["data"]):
    new_datas.append({
        "id": str(data_id).zfill(10),
        "question": data["question"],
        "labels": data["answers"],
        "image_path": f"{data['image_id']}.jpg",
    })

with open(backup, "w") as f:
    json.dump(new_datas, f, indent=4)

small = new_datas[:3]
with open(dst, "w") as f:
    json.dump(small, f, indent=4)

print("Prepared", len(small), "examples")
print(small[0])


Prepared 3 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [16]:
!mkdir -p /content/mllms_know/data/results
!rm -f /content/mllms_know/data/results/blip-textvqa-rel_att.json


In [17]:
!python run.py --task textvqa --model blip --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 16:02:52.406126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774800172.432062   10544 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774800172.439665   10544 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774800172.689413   10544 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800172.689462   10544 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800172.689467   10544 computation_placer.cc:177] computation placer alr

In [19]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 3/3 [00:00<00:00, 1527.79it/s]
  model_name     task   method  raw_acc   crop_acc
0       blip  textvqa  rel_att      0.0  33.333333


In [20]:
import json

with open("/content/mllms_know/evaluation_report.json") as f:
    report = json.load(f)

[x for x in report if x["model_name"] == "blip" and x["task"] == "textvqa" and x["method"] == "rel_att"]


[{'model_name': 'blip',
  'task': 'textvqa',
  'method': 'rel_att',
  'raw_acc': 0.0,
  'crop_acc': 33.333333333333336}]

First 5 Examples

In [21]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [22]:
import json

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:5]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Prepared", len(small_data), "examples")
print(small_data[0])


Prepared 5 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [23]:
!rm -f /content/mllms_know/data/results/blip-textvqa-rel_att.json


In [24]:
!python run.py --task textvqa --model blip --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 16:12:39.147192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774800759.173889   13128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774800759.182079   13128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774800759.212639   13128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800759.212669   13128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800759.212673   13128 computation_placer.cc:177] computation placer alr

In [25]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 5/5 [00:00<00:00, 1829.02it/s]
  model_name     task   method  raw_acc  crop_acc
0       blip  textvqa  rel_att     20.0      40.0


In [26]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [27]:
import json

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:10]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Prepared", len(small_data), "examples")
print(small_data[0])


Prepared 10 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [28]:
!rm -f /content/mllms_know/data/results/blip-textvqa-rel_att.json


In [29]:
!python run.py --task textvqa --model blip --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 16:18:50.992972: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774801131.019285   14808 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774801131.027203   14808 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774801131.057777   14808 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774801131.057811   14808 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774801131.057815   14808 computation_placer.cc:177] computation placer alr

In [30]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 10/10 [00:00<00:00, 1533.17it/s]
  model_name     task   method  raw_acc  crop_acc
0       blip  textvqa  rel_att     10.0      30.0


Grad-Attention

In [31]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [32]:
import shutil
import json

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:3]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Prepared", len(small_data), "examples")
print(small_data[0])


Prepared 3 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [35]:
!rm -f /content/mllms_know/data/results/blip-textvqa-rel_att.json


In [36]:
!rm -f /content/mllms_know/data/results/blip-textvqa-grad_att.json
!python run.py --task textvqa --model blip --method grad_att --save_path /content/mllms_know/data/results


2026-03-29 16:33:34.089682: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774802014.116100   18718 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774802014.124082   18718 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774802014.155708   18718 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774802014.155751   18718 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774802014.155755   18718 computation_placer.cc:177] computation placer alr

In [37]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 3/3 [00:00<00:00, 1786.33it/s]
  model_name     task    method  raw_acc  crop_acc
0       blip  textvqa  grad_att      0.0       0.0


In [38]:
import shutil
import json

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:5]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Prepared", len(small_data), "examples")
print(small_data[0])


Prepared 5 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [39]:
!rm -f /content/mllms_know/data/results/blip-textvqa-grad_att.json


In [40]:
!python run.py --task textvqa --model blip --method grad_att --save_path /content/mllms_know/data/results


2026-03-29 16:58:59.948534: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774803539.974892   25200 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774803539.984464   25200 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774803540.017434   25200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774803540.017477   25200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774803540.017482   25200 computation_placer.cc:177] computation placer alr

In [41]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 5/5 [00:00<00:00, 1838.80it/s]
  model_name     task    method  raw_acc  crop_acc
0       blip  textvqa  grad_att     20.0      20.0
